# Classification de Nuages de Points 3D
## Graph CNN · Apprentissage Fédéré · Distillation de Connaissances

**Architectures :** TACO-Net (Graph CNN personnalisé) · PointGCN  
**Dataset :** Synthétique (pas de téléchargement)  
**Environnement :** Kaggle GPU (T4 × 2 ou P100)

---
### Table des matières
1. Installation & Imports
2. Configuration globale
3. Génération du Dataset Synthétique
4. Pipeline de Données
5. Architectures Graph CNN (TACO-Net & PointGCN)
6. Partie 1 – Baseline Centralisée
7. Partie 2 – Apprentissage Fédéré (HFL + VFL)
8. Partie 3 – Distillation de Connaissances
9. Comparaison & Analyse Finale

---
## 1. Installation des dépendances (Kaggle compatible)

In [ ]:
import subprocess, sys, torch

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Kaggle fournit déjà torch 2.4.0 avec CUDA 12.1, nous n’installons pas torch.
# Installation de PyTorch Geometric adaptée à la version de torch présente.
torch_version = torch.__version__.split('+')[0]
cuda_suffix = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'Installation PyG pour torch {torch_version}+{cuda_suffix}')

pip_install('torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv', 'torch-geometric',
            '-f', f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_suffix}.html')

pip_install('matplotlib', 'seaborn', 'scikit-learn', 'pandas', 'numpy', 'tqdm')

print('✅ Dépendances installées. Veuillez redémarrer le kernel si des erreurs d’import subsistent.')

In [ ]:
# ─── Imports principaux ───────────────────────────────────────────────────────
import os, copy, random, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import knn_graph, global_mean_pool, global_max_pool, GCNConv, EdgeConv
from torch_geometric.loader import DataLoader as PyGDataLoader

warnings.filterwarnings('ignore')
print(f'PyTorch : {torch.__version__}')
print(f'PyG     : {torch_geometric.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

---
## 2. Configuration Globale (réglée pour Kaggle)

In [ ]:
# ─── Seed & Device ────────────────────────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ─── Chemins (Kaggle working directory) ───────────────────────────────────────
ROOT     = Path('/kaggle/working')
CKPT_DIR = ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Hyperparamètres Dataset ──────────────────────────────────────────────────
NUM_POINTS        = 512        # Points par objet (réduit pour mémoire)
NUM_CLASSES       = 10         # Classes (synthétiques)
TOTAL_SAMPLES     = 1200       # 120 objets par classe
TRAIN_RATIO       = 0.70
VAL_RATIO         = 0.15

# ─── Hyperparamètres Entraînement ─────────────────────────────────────────────
BATCH_SIZE     = 16
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
EPOCHS_C       = 5          # Bas pour test rapide
K_NEIGHBORS    = 16

# ─── Hyperparamètres Fédéré ───────────────────────────────────────────────────
NUM_ROUNDS     = 2
LOCAL_EPOCHS   = 1
NUM_CLIENTS_H  = 4

# ─── Hyperparamètres Distillation ─────────────────────────────────────────────
ALPHA          = 0.7
TEMPERATURE    = 4.0
EPOCHS_KD      = 5

print(f'✅ Configuration chargée (Device: {DEVICE})')
print(f'   Points/objet : {NUM_POINTS}, Classes: {NUM_CLASSES}, Total {TOTAL_SAMPLES} objets')

---
## 3. Dataset Synthétique
On génère des nuages de points aléatoires (x,y,z,r,g,b) pour éviter tout téléchargement.
Chaque classe est représentée par une distribution légèrement différente pour rendre la tâche non triviale.

In [ ]:
def generate_synthetic_data(num_samples, num_points, num_classes, noise=0.1):
    """Génère des nuages de points 3D + couleur, étiquetés de 0 à num_classes-1."""
    data_list = []
    for i in range(num_samples):
        label = i % num_classes
        # Chaque classe a un centre différent dans l'espace
        center_xyz = torch.tensor([np.sin(label)*2.0, np.cos(label)*2.0, label*0.5], dtype=torch.float)
        center_rgb = torch.tensor([(label%3)*0.8, ((label+1)%3)*0.8, ((label+2)%3)*0.8], dtype=torch.float)
        # Nuage de points bruité autour du centre
        pos = center_xyz.unsqueeze(0) + torch.randn(num_points, 3) * noise
        color = torch.clamp(center_rgb.unsqueeze(0) + torch.randn(num_points, 3) * noise, 0.0, 1.0)
        x = torch.cat([pos, color], dim=1)  # shape (N,6)
        # Construction du graphe k-NN
        edge_index = knn_graph(pos, k=K_NEIGHBORS, loop=False)
        data = Data(x=x, pos=pos, color=color, edge_index=edge_index,
                    y=torch.tensor([label], dtype=torch.long))
        data_list.append(data)
    return data_list

set_seed()
all_data = generate_synthetic_data(TOTAL_SAMPLES, NUM_POINTS, NUM_CLASSES)

# Split train/val/test
indices = list(range(TOTAL_SAMPLES))
train_end = int(TOTAL_SAMPLES * TRAIN_RATIO)
val_end = int(TOTAL_SAMPLES * (TRAIN_RATIO + VAL_RATIO))
train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

class ListDataset(Dataset):
    def __init__(self, data_list, indices):
        self.samples = [data_list[i] for i in indices]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

ds_train = ListDataset(all_data, train_idx)
ds_val   = ListDataset(all_data, val_idx)
ds_test  = ListDataset(all_data, test_idx)

loader_train = PyGDataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
loader_val   = PyGDataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False)
loader_test  = PyGDataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)

print(f'Données : {len(ds_train)} train, {len(ds_val)} val, {len(ds_test)} test')
print(f'Batchs : {len(loader_train)} train, {len(loader_val)} val, {len(loader_test)} test')

# Visualisation rapide
sample_batch = next(iter(loader_train))
fig, axes = plt.subplots(1, 3, figsize=(12,4), subplot_kw={'projection':'3d'})
for i, ax in enumerate(axes):
    mask = sample_batch.batch == i
    pts = sample_batch.pos[mask].cpu().numpy()
    col = sample_batch.color[mask].cpu().numpy()
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=col, s=0.5)
    ax.set_title(f'Classe {sample_batch.y[i].item()}')
    ax.set_axis_off()
plt.suptitle('Exemples synthétiques', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Architectures Graph CNN
TACO‑Net avec attention par graphe corrigée, et PointGCN standard.

In [ ]:
# ─── Bloc TACO avec attention topologique par graphe ──────────────────────────
class TACOBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = EdgeConv(
            nn=nn.Sequential(
                nn.Linear(2 * in_ch, out_ch),
                nn.BatchNorm1d(out_ch),
                nn.LeakyReLU(0.2),
                nn.Linear(out_ch, out_ch),
                nn.BatchNorm1d(out_ch),
                nn.LeakyReLU(0.2),
            ),
            aggr='max'
        )
        self.attn = nn.Sequential(
            nn.Linear(out_ch, out_ch // 4),
            nn.ReLU(),
            nn.Linear(out_ch // 4, out_ch),
            nn.Sigmoid()
        )
        self.skip = nn.Linear(in_ch, out_ch) if in_ch != out_ch else nn.Identity()
        self.bn = nn.BatchNorm1d(out_ch)

    def forward(self, x, edge_index, batch):
        # h : features nodales après EdgeConv
        h = self.edge_conv(x, edge_index)
        # Moyenne globale par graphe pour le gate
        gate = self.attn(h)
        mean_per_graph = global_mean_pool(gate, batch)[batch]  # broadcast par nœud
        gate = gate * mean_per_graph
        h = h * gate
        return F.leaky_relu(self.bn(h + self.skip(x)), 0.2)

class TACONet(nn.Module):
    def __init__(self, in_ch=6, num_cls=10, widths=[32, 64, 128]):  # réduit pour vélocité
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_ch, widths[0]),
            nn.BatchNorm1d(widths[0]),
            nn.LeakyReLU(0.2)
        )
        self.blocks = nn.ModuleList()
        dims = [widths[0]] + widths
        for i in range(len(widths)):
            self.blocks.append(TACOBlock(dims[i], dims[i+1]))
        total_ch = widths[0] + sum(widths)
        self.fusion = nn.Sequential(
            nn.Linear(total_ch, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2)
        )
        self.head = nn.Sequential(
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        skips = [h]
        for block in self.blocks:
            h = block(h, edge_index, batch)
            skips.append(h)
        multi = torch.cat(skips, dim=-1)
        feat = self.fusion(multi)
        g_mean = global_mean_pool(feat, batch)
        g_max = global_max_pool(feat, batch)
        return self.head(torch.cat([g_mean, g_max], dim=-1))

    def get_embeddings(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        skips = [h]
        for block in self.blocks:
            h = block(h, edge_index, batch)
            skips.append(h)
        multi = torch.cat(skips, dim=-1)
        feat = self.fusion(multi)
        return torch.cat([global_mean_pool(feat, batch), global_max_pool(feat, batch)], dim=-1)

# ─── PointGCN (structure réduite) ─────────────────────────────────────────────
class PointGCN(nn.Module):
    def __init__(self, in_ch=6, num_cls=10, hidden=64):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(in_ch, hidden), nn.BatchNorm1d(hidden), nn.ReLU())
        self.gcn1 = GCNConv(hidden, hidden)
        self.gcn2 = GCNConv(hidden, hidden*2)
        self.gcn3 = GCNConv(hidden*2, hidden*2)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.bn2 = nn.BatchNorm1d(hidden*2)
        self.bn3 = nn.BatchNorm1d(hidden*2)
        self.skip2 = nn.Linear(hidden, hidden*2)
        self.head = nn.Sequential(
            nn.Linear(hidden*2*2, 128),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        h = F.relu(self.bn1(self.gcn1(h, edge_index))) + h
        h = F.relu(self.bn2(self.gcn2(h, edge_index))) + self.skip2(h)
        h = F.relu(self.bn3(self.gcn3(h, edge_index)))
        g = torch.cat([global_max_pool(h, batch), global_mean_pool(h, batch)], dim=-1)
        return self.head(g)

    def get_embeddings(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        h = F.relu(self.bn1(self.gcn1(h, edge_index))) + h
        h = F.relu(self.bn2(self.gcn2(h, edge_index))) + self.skip2(h)
        h = F.relu(self.bn3(self.gcn3(h, edge_index)))
        return torch.cat([global_max_pool(h, batch), global_mean_pool(h, batch)], dim=-1)

# Test rapide
net_taco = TACONet().to(DEVICE)
net_gcn  = PointGCN().to(DEVICE)
with torch.no_grad():
    out_t = net_taco(sample_batch.to(DEVICE))
    out_g = net_gcn(sample_batch.to(DEVICE))
print(f'TACO-Net out: {out_t.shape} | params: {sum(p.numel() for p in net_taco.parameters()):,}')
print(f'PointGCN out: {out_g.shape} | params: {sum(p.numel() for p in net_gcn.parameters()):,}')

---
## 5. Fonctions d’entraînement et d’évaluation

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        correct += (logits.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        loss = criterion(logits, batch.y)
        total_loss += loss.item() * batch.num_graphs
        correct += (logits.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_full(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        preds = model(batch).argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch.y.cpu().tolist())
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = (all_preds == all_labels).mean()
    cm = confusion_matrix(all_labels, all_preds, labels=range(NUM_CLASSES))
    return acc, cm

def train_model(model, loader_tr, loader_va, epochs, ckpt_name, device, lr=LR, wd=WEIGHT_DECAY):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_val_acc = 0.0
    ckpt_path = CKPT_DIR / f'{ckpt_name}_best.pt'
    for epoch in range(1, epochs+1):
        t_loss, t_acc = train_epoch(model, loader_tr, optimizer, criterion, device)
        v_loss, v_acc = eval_epoch(model, loader_va, criterion, device)
        scheduler.step()
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), ckpt_path)
        if epoch % 2 == 0 or epoch == 1:
            print(f'  Ep {epoch:2d}/{epochs} | train={t_acc:.4f}, val={v_acc:.4f}')
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return best_val_acc

print('✅ Fonctions d\'entraînement prêtes.')

---
## 6. Partie 1 – Baseline Centralisée (C1)

In [ ]:
print('='*60)
print('C1 — Baseline Centralisée : TACO-Net')
print('='*60)
set_seed()
model_taco_c1 = TACONet().to(DEVICE)
best_taco_c1 = train_model(model_taco_c1, loader_train, loader_val, EPOCHS_C, 'taco_c1', DEVICE)

print('\n' + '='*60)
print('C1 — Baseline Centralisée : PointGCN')
print('='*60)
set_seed()
model_gcn_c1 = PointGCN().to(DEVICE)
best_gcn_c1 = train_model(model_gcn_c1, loader_train, loader_val, EPOCHS_C, 'gcn_c1', DEVICE)

acc_taco_c1, _ = evaluate_full(model_taco_c1, loader_test, DEVICE)
acc_gcn_c1, _  = evaluate_full(model_gcn_c1, loader_test, DEVICE)
print(f'\n📊 C1 Test : TACO={acc_taco_c1:.4f}, PointGCN={acc_gcn_c1:.4f}')

---
## 7. Partie 2 – Apprentissage Fédéré

In [ ]:
# Partitionnement des données pour HFL (IID et non‑IID)
def make_iid_split(indices, n_clients=4):
    np.random.shuffle(indices)
    splits = np.array_split(indices, n_clients)
    return {i: split.tolist() for i, split in enumerate(splits)}

def make_non_iid_split(indices, labels, n_clients=4):
    # Répartit de sorte que chaque client voit un sous-ensemble de classes
    clients = {i: [] for i in range(n_clients)}
    sorted_by_label = sorted(zip(indices, labels), key=lambda x: x[1])
    for idx, (idx_, lbl) in enumerate(sorted_by_label):
        client_id = idx % n_clients
        clients[client_id].append(idx_)
    return clients

all_labels = np.array([d.y.item() for d in all_data])
client_iid = make_iid_split(train_idx, NUM_CLIENTS_H)
client_niid = make_non_iid_split(train_idx, all_labels, NUM_CLIENTS_H)

print(f'IID clients sizes: {[len(v) for v in client_iid.values()]}')
print(f'Non-IID clients sizes: {[len(v) for v in client_niid.values()]}')

In [ ]:
def fed_avg(state_dicts, sizes):
    total = sum(sizes)
    weights = [s / total for s in sizes]
    avg_sd = {}
    for key in state_dicts[0].keys():
        tensors = [sd[key].float() for sd in state_dicts]
        stacked = torch.stack(tensors, dim=0)
        w = torch.tensor(weights, dtype=torch.float32, device=stacked.device)
        shape = [-1] + [1]*(stacked.dim()-1)
        avg_sd[key] = (stacked * w.view(*shape)).sum(0).type(state_dicts[0][key].dtype)
    return avg_sd

def client_local_train(model_class, global_sd, client_indices, local_epochs, device, **kwargs):
    model = model_class(**kwargs).to(device)
    model.load_state_dict(copy.deepcopy(global_sd))
    ds = ListDataset(all_data, client_indices)
    loader = PyGDataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    for _ in range(local_epochs):
        for batch in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch), batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return model.state_dict(), len(ds)

def federated_training(model_class, client_data, n_rounds, ckpt_name, device, **kwargs):
    global_model = model_class(**kwargs).to(device)
    global_sd = global_model.state_dict()
    best_val_acc = 0.0
    ckpt_path = CKPT_DIR / f'{ckpt_name}_best.pt'
    criterion = nn.CrossEntropyLoss()
    for rnd in range(1, n_rounds+1):
        client_sds, sizes = [], []
        for cid, indices in client_data.items():
            sd, sz = client_local_train(model_class, global_sd, indices, LOCAL_EPOCHS, device, **kwargs)
            client_sds.append(sd)
            sizes.append(sz)
        global_sd = fed_avg(client_sds, sizes)
        global_model.load_state_dict(global_sd)
        _, v_acc = eval_epoch(global_model, loader_val, criterion, device)
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(global_sd, ckpt_path)
        print(f'  Round {rnd:2d}/{n_rounds} | val_acc={v_acc:.4f}')
    global_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return global_model, best_val_acc

# HFL IID (C2)
print('='*60)
print('C2 — HFL IID : TACO-Net')
print('='*60)
model_taco_c2, _ = federated_training(TACONet, client_iid, NUM_ROUNDS, 'taco_c2', DEVICE)

print('\n' + '='*60)
print('C2 — HFL IID : PointGCN')
print('='*60)
model_gcn_c2, _ = federated_training(PointGCN, client_iid, NUM_ROUNDS, 'gcn_c2', DEVICE)

# HFL non‑IID (C3)
print('='*60)
print('C3 — HFL non‑IID : TACO-Net')
print('='*60)
model_taco_c3, _ = federated_training(TACONet, client_niid, NUM_ROUNDS, 'taco_c3', DEVICE)

print('\n' + '='*60)
print('C3 — HFL non‑IID : PointGCN')
print('='*60)
model_gcn_c3, _ = federated_training(PointGCN, client_niid, NUM_ROUNDS, 'gcn_c3', DEVICE)

acc_taco_c2, _ = evaluate_full(model_taco_c2, loader_test, DEVICE)
acc_gcn_c2, _  = evaluate_full(model_gcn_c2, loader_test, DEVICE)
acc_taco_c3, _ = evaluate_full(model_taco_c3, loader_test, DEVICE)
acc_gcn_c3, _  = evaluate_full(model_gcn_c3, loader_test, DEVICE)
print(f'\n📊 C2 IID     : TACO={acc_taco_c2:.4f}, GCN={acc_gcn_c2:.4f}')
print(f'📊 C3 non‑IID : TACO={acc_taco_c3:.4f}, GCN={acc_gcn_c3:.4f}')

---
## 8. Partie 2 – Fédération Verticale (VFL)

In [ ]:
class VFLEncoderXYZ(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(3, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2))
        self.block1 = TACOBlock(64, 128)
        self.block2 = TACOBlock(128, 128)
        self.proj = nn.Sequential(nn.Linear(256, emb_dim), nn.BatchNorm1d(emb_dim), nn.ReLU())

    def forward(self, data):
        x = data.pos
        edge_index, batch = data.edge_index, data.batch
        h = self.stem(x)
        h = self.block1(h, edge_index, batch)
        h = self.block2(h, edge_index, batch)
        g = torch.cat([global_max_pool(h, batch), global_mean_pool(h, batch)], dim=-1)
        return self.proj(g)

class VFLEncoderRGB(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(3, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gcn1 = GCNConv(64, 128)
        self.gcn2 = GCNConv(128, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(128)
        self.proj = nn.Sequential(nn.Linear(256, emb_dim), nn.BatchNorm1d(emb_dim), nn.ReLU())

    def forward(self, data):
        x = data.color
        edge_index, batch = data.edge_index, data.batch
        h = self.stem(x)
        h = F.relu(self.bn1(self.gcn1(h, edge_index)))
        h = F.relu(self.bn2(self.gcn2(h, edge_index)))
        g = torch.cat([global_max_pool(h, batch), global_mean_pool(h, batch)], dim=-1)
        return self.proj(g)

class VFLServerFusion(nn.Module):
    def __init__(self, emb_dim=128, num_cls=10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(emb_dim*2, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_cls)
        )

    def forward(self, eA, eB):
        return self.fc(torch.cat([eA, eB], dim=-1))

def train_vfl(loader_tr, loader_va, epochs, ckpt_name, device):
    encA = VFLEncoderXYZ().to(device)
    encB = VFLEncoderRGB().to(device)
    server = VFLServerFusion().to(device)
    params = list(encA.parameters()) + list(encB.parameters()) + list(server.parameters())
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0
    ckpt_path = CKPT_DIR / f'{ckpt_name}_best.pt'
    for epoch in range(1, epochs+1):
        encA.train(); encB.train(); server.train()
        correct, total = 0, 0
        for batch in loader_tr:
            batch = batch.to(device)
            eA = encA(batch)
            eB = encB(batch)
            logits = server(eA, eB)
            loss = criterion(logits, batch.y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()
            correct += (logits.argmax(1) == batch.y).sum().item()
            total += batch.num_graphs
        v_acc, _ = evaluate_full_server(encA, encB, server, loader_va, device) if False else 0
        # Évaluation simplifiée
        encA.eval(); encB.eval(); server.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for batch in loader_va:
                batch = batch.to(device)
                logits = server(encA(batch), encB(batch))
                v_correct += (logits.argmax(1) == batch.y).sum().item()
                v_total += batch.num_graphs
        v_acc = v_correct / v_total
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save({'encA': encA.state_dict(), 'encB': encB.state_dict(), 'server': server.state_dict()}, ckpt_path)
        if epoch % 2 == 0 or epoch == 1:
            print(f'  VFL Ep {epoch:2d}/{epochs} | train={correct/total:.4f}, val={v_acc:.4f}')
    best = torch.load(ckpt_path, map_location=device)
    encA.load_state_dict(best['encA'])
    encB.load_state_dict(best['encB'])
    server.load_state_dict(best['server'])
    return encA, encB, server, best_val_acc

print('='*60)
print('VFL — Fédération Verticale')
print('='*60)
encA_vfl, encB_vfl, server_vfl, _ = train_vfl(loader_train, loader_val, EPOCHS_C, 'vfl', DEVICE)

@torch.no_grad()
def eval_vfl(encA, encB, server, loader, device):
    encA.eval(); encB.eval(); server.eval()
    correct, total = 0, 0
    for batch in loader:
        batch = batch.to(device)
        logits = server(encA(batch), encB(batch))
        correct += (logits.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
    return correct / total

vfl_acc = eval_vfl(encA_vfl, encB_vfl, server_vfl, loader_test, DEVICE)
print(f'\n📊 VFL Test Accuracy : {vfl_acc:.4f}')

---
## 9. Partie 3 – Distillation de Connaissances (C4)

In [ ]:
# Élèves avec largeurs réduites (compression >4x)
class TACONetSmall(nn.Module):
    def __init__(self, in_ch=6, num_cls=10, widths=[8, 16, 32]):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(in_ch, widths[0]), nn.BatchNorm1d(widths[0]), nn.LeakyReLU(0.2))
        self.blocks = nn.ModuleList()
        dims = [widths[0]] + widths
        for i in range(len(widths)):
            self.blocks.append(TACOBlock(dims[i], dims[i+1]))
        total_ch = widths[0] + sum(widths)
        self.fusion = nn.Sequential(nn.Linear(total_ch, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2))
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(64, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        skips = [h]
        for block in self.blocks:
            h = block(h, edge_index, batch)
            skips.append(h)
        multi = torch.cat(skips, dim=-1)
        feat = self.fusion(multi)
        g = torch.cat([global_mean_pool(feat, batch), global_max_pool(feat, batch)], dim=-1)
        return self.head(g)

class PointGCNSmall(nn.Module):
    def __init__(self, in_ch=6, num_cls=10, hidden=16):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(in_ch, hidden), nn.BatchNorm1d(hidden), nn.ReLU())
        self.gcn1 = GCNConv(hidden, hidden)
        self.gcn2 = GCNConv(hidden, hidden*2)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.bn2 = nn.BatchNorm1d(hidden*2)
        self.skip = nn.Linear(hidden, hidden*2)
        self.head = nn.Sequential(
            nn.Linear(hidden*2*2, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_cls)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.stem(x)
        h = F.relu(self.bn1(self.gcn1(h, edge_index))) + h
        h = F.relu(self.bn2(self.gcn2(h, edge_index))) + self.skip(h)
        g = torch.cat([global_max_pool(h, batch), global_mean_pool(h, batch)], dim=-1)
        return self.head(g)

def distillation_loss(logits_s, logits_t, labels, alpha=ALPHA, tau=TEMPERATURE):
    logits_t = logits_t.detach()
    loss_ce = F.cross_entropy(logits_s, labels)
    soft_s = F.log_softmax(logits_s / tau, dim=-1)
    soft_t = F.softmax(logits_t / tau, dim=-1)
    loss_kl = F.kl_div(soft_s, soft_t, reduction='batchmean') * (tau**2)
    return (1 - alpha) * loss_ce + alpha * loss_kl

def train_kd(teacher, student, loader_tr, loader_va, epochs, ckpt_name, device):
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False
    optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_val_acc = 0.0
    ckpt_path = CKPT_DIR / f'{ckpt_name}_best.pt'
    for epoch in range(1, epochs+1):
        student.train()
        correct, total = 0, 0
        for batch in loader_tr:
            batch = batch.to(device)
            with torch.no_grad():
                logits_t = teacher(batch)
            logits_s = student(batch)
            loss = distillation_loss(logits_s, logits_t, batch.y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            correct += (logits_s.argmax(1) == batch.y).sum().item()
            total += batch.num_graphs
        v_loss, v_acc = eval_epoch(student, loader_va, nn.CrossEntropyLoss(), device)
        scheduler.step()
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(student.state_dict(), ckpt_path)
        if epoch % 2 == 0 or epoch == 1:
            print(f'  KD Ep {epoch:2d}/{epochs} | train={correct/total:.4f}, val={v_acc:.4f}')
    student.load_state_dict(torch.load(ckpt_path, map_location=device))
    return student

print('='*60)
print('C4 — Distillation TACO‑Net')
print('='*60)
student_taco = TACONetSmall().to(DEVICE)
student_taco = train_kd(model_taco_c1, student_taco, loader_train, loader_val, EPOCHS_KD, 'kd_taco', DEVICE)

print('\n' + '='*60)
print('C4 — Distillation PointGCN')
print('='*60)
student_gcn = PointGCNSmall().to(DEVICE)
student_gcn = train_kd(model_gcn_c1, student_gcn, loader_train, loader_val, EPOCHS_KD, 'kd_gcn', DEVICE)

acc_st_taco, _ = evaluate_full(student_taco, loader_test, DEVICE)
acc_st_gcn, _  = evaluate_full(student_gcn, loader_test, DEVICE)
print(f'\n📊 C4 Distillation : TACO student={acc_st_taco:.4f}, GCN student={acc_st_gcn:.4f}')

---
## 10. Comparaison et Analyse Finale

In [ ]:
# Récapitulatif des précisions (test)
results = {
    'C1 Baseline':     (acc_taco_c1, acc_gcn_c1),
    'C2 HFL IID':      (acc_taco_c2, acc_gcn_c2),
    'C3 HFL non‑IID':  (acc_taco_c3, acc_gcn_c3),
    'C4 Distillation': (acc_st_taco, acc_st_gcn)
}

df_res = pd.DataFrame(results, index=['TACO-Net', 'PointGCN']).T
print('📊 Résultats finaux (test accuracy) :')
print(df_res.to_string())

# Graphique
ax = df_res.plot(kind='bar', figsize=(10,6), color=['steelblue','darkorange'])
ax.set_ylabel('Accuracy')
ax.set_title('Comparaison des configurations')
ax.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge')
plt.tight_layout()
plt.savefig(ROOT / 'comparison.png', dpi=100)
plt.show()

print(f'\n✅ Notebook terminé. VFL Accuracy : {vfl_acc:.4f}')